# Hands-on 31 (Bab 12): Membuat Alert Sederhana jika Skor Anomali Tinggi

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 12.

Sistem IoT-ML yang baik tidak berhenti pada prediksi; ia mendorong tindakan. Di sini
subscriber pipeline menerbitkan alert ke topik `iotlab/room01/alert` bila model
memprediksi FAULT, atau suhu > 32 C dan CO2 > 1200 ppm bersamaan. Alert juga disimpan
ke `outputs/reports/pipeline_room_alerts.csv` (InfluxDB opsional via `--influx`).

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Jalankan Pipeline + Alert (broker, subscriber+alert, listener alert, simulator)

In [2]:
import subprocess, sys, time

broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = subprocess.Popen(
        [sys.executable, "scripts/pipeline_room_with_alert.py",
         "--max-messages", "12", "--timeout", "45"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    listener = subprocess.Popen(
        [sys.executable, "scripts/mqtt/subscribe_sensor.py",
         "--topic", "iotlab/room01/alert", "--max-messages", "1", "--timeout", "45"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time.sleep(1)
    subprocess.run([sys.executable, "scripts/publisher_room_simulator.py",
                    "--count", "12", "--interval", "0.3"],
                   capture_output=True, text=True)
    out_sub, _ = sub.communicate()
    print("--- Output subscriber+alert ---")
    print(out_sub)
    out_listener, _ = listener.communicate()
    print("--- Output listener topik alert ---")
    print(out_listener[:700])
finally:
    broker.terminate()

--- Output subscriber+alert ---
[INFO] Storage alert: CSV lokal outputs/reports/pipeline_room_alerts.csv
[INFO] Subscriber+Alert terhubung ke broker.
[INFO] Subscribe ke: iotlab/room01/telemetry
[2026-10-22T12:04:17] -> status=normal
[2026-10-22T12:04:18] -> status=normal
[2026-10-22T12:04:18] -> status=normal
[2026-10-22T12:04:18] -> status=normal
[2026-10-22T12:04:19] -> status=warning
[2026-10-22T12:04:19] -> status=warning
[2026-10-22T12:04:19] -> status=normal
[2026-10-22T12:04:20] -> status=warning
[2026-10-22T12:04:20] -> status=normal
[2026-10-22T12:04:20] -> status=fault
[PERINGATAN] ALERT: {"timestamp": "2026-10-22T12:04:20", "device_id": "room01", "pred_status": "fault", "alert_reason": "Model memprediksi FAULT", "temperature_c": 25.46, "humidity_pct": 52.42, "co2_ppm": 1230, "power_watt": 259.06}
[2026-10-22T12:04:21] -> status=fault
[PERINGATAN] ALERT: {"timestamp": "2026-10-22T12:04:21", "device_id": "room01", "pred_status": "fault", "alert_reason": "Model memprediksi FAU

## 2. Verifikasi Log Alert

In [3]:
import pandas as pd
alerts = pd.read_csv("outputs/reports/pipeline_room_alerts.csv")
print(alerts.to_string(index=False) if len(alerts) else "Tidak ada alert pada batch ini.")

          timestamp device_id pred_status            alert_reason  temperature_c  humidity_pct  co2_ppm  power_watt
2026-10-22T12:04:20    room01       fault Model memprediksi FAULT          25.46         52.42     1230      259.06
2026-10-22T12:04:21    room01       fault Model memprediksi FAULT          31.77         58.91     1407      142.85


## Penutup

Alert berbasis status model (FAULT) dan aturan sensor sederhana (suhu + CO2 tinggi)
diterbitkan ke topik terpisah dan tersimpan di storage. Pola ini mudah dikembangkan:
tambahkan ambang skor anomali, kirim ke notifikasi, atau tampilkan panel alert di
Grafana sesuai buku.

## Tambahan: Versi Sederhana Berbasis Skor Anomali

Pendekatan B buku: skor 0.4 (suhu > 31) + 0.4 (CO2 > 1100) + 0.2 (daya > 220), alert bila skor >= 0.7. Dijalankan pada dataset sintetis ruangan.

In [4]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/alert_score_demo.py"], capture_output=True, text=True)
print(r.stdout)

Total baris : 2160
Alert (skor >= 0.7): 18 (0.83%)
Distribusi skor:
0.0    2119
0.2       5
0.4       6
0.6      12
0.8       1
1.0      17

Contoh alert:
          timestamp device_id  temperature_c  humidity_pct  co2_ppm  power_watt status_label  is_anomaly
2026-01-01 07:10:00    room01          33.20         66.37     1317      277.23        fault           1
2026-01-02 17:40:00    room01          34.26         71.19     1278      332.34        fault           1
2026-01-03 02:20:00    room01          31.14         71.07     1153      185.24        fault           1
2026-01-04 09:40:00    room01          34.45         64.95     1698      276.83        fault           1
2026-01-04 17:40:00    room01          32.65         64.33     1352      299.82        fault           1



## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 33 dalam urutan buku) sebagai panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 9. Langkah kerja yang harus dilakukan pembaca

Hands-on 30
1. jalankan Mosquitto, InfluxDB, dan Grafana
2. jalankan simulator sensor
3. jalankan subscriber Python
4. validasi data masuk ke broker
5. pastikan preprocessing dan prediksi berjalan
6. periksa data dan hasil inferensi di InfluxDB
7. bangun dashboard Grafana

Hands-on 31
1. tambahkan logika alert ke subscriber
2. tentukan aturan alert sederhana
3. kirim alert ke topik MQTT khusus
4. simpan alert ke measurement khusus
5. tampilkan alert di dashboard
6. uji dengan nilai sensor ekstrem atau simulasi fault